# **Installs & Imports**

In [ ]:
!pip install transformers
!pip install transformers[torch]
!pip install accelerate -U
!pip install accelerate
!pip install biopython
!pip install -U sacremoses
!pip install openai
!pip install openai==1.55.3 httpx==0.27.2

from transformers import pipeline, set_seed, BioGptTokenizer, BioGptForCausalLM, Trainer, TrainingArguments, TextDataset, DataCollatorForLanguageModeling
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config, AutoTokenizer, AutoModelForCausalLM, AutoModel, GPT2LMHeadModel, T5ForConditionalGeneration
import os
import urllib.parse
import requests, time
import xml.etree.ElementTree as ET
import io
from requests.exceptions import ChunkedEncodingError, RequestException
from urllib.parse import quote_plus
from io import StringIO
from time import sleep
from Bio import Entrez

# **Generate MeSH Terms & Keywords**

In [ ]:
def preprocess_titles():
  folder_path = '/content/query_files'
  topic_numbers = []
  titles = []

  try:
      files_in_folder = os.listdir(folder_path)

      for i, filename in enumerate(files_in_folder):
          file_path = os.path.join(folder_path, filename)

          with open(file_path, 'r', encoding='utf-8') as file:
              topic_number = file.readline().strip()
              title = file.readline().strip()
              topic_numbers.append(topic_number)
              titles.append(title)

  except FileNotFoundError:
      print(f"Folder not found: {folder_path}")
  except Exception as e:
      print(f"An error occurred: {e}")

  return topic_numbers, titles

def extract_terms(line):
  mesh_terms_str = line
  mesh_terms = [term.strip().strip("'") for term in mesh_terms_str.strip("[]").split(',')]
  return(mesh_terms)

def remove_duplicate_terms(strings):
  seen = set()
  unique_strings = []
  for string in strings:
      if string not in seen:
          unique_strings.append(string)
          seen.add(string)
  return unique_strings

def fetch_mesh_terms(term, api_key):
  encoded_term = quote_plus(term)
  encoded_term += "[mesh]"
  mesh_id = None
  search_url = f'http://eutils.ncbi.nlm.nih.gov/entrez//eutils/esearch.fcgi/' + \
                f'?db=mesh' + \
                f'&term={encoded_term}' + \
                f'&retmode=json' + \
                f'&sort=relevance' + \
                f'&retmax=1' +\
                f'&api_key={api_key}'

  try:
      response = requests.get(search_url)
      response.raise_for_status()
      summary = response.json()
      if 'esearchresult' in summary and 'idlist' in summary['esearchresult']:
        if len(summary['esearchresult']['idlist']) > 0:
          mesh_id = summary['esearchresult']['idlist'][0]
  except HTTPError as http_err:
      mesh_id = None
      print(f'HTTP error occurred: {http_err}')
  except RequestException as err:
      mesh_id = None
      print(f'Other error occurred: {err}')
  except Exception as exc:
      mesh_id = None
      print(f'Unexpected error occurred: {exc}')

  return mesh_id

**Generate MeSH Terms**

In [ ]:
api_key = '317014466decf6d06f198a0903107ca95408' #Take this out before uploading
topic_ids, titles = preprocess_titles()

MeSHGenerating_tokenizer = BioGptTokenizer.from_pretrained('/content/drive/MyDrive/models/BioGPT1000_MeSHTerms_Model') #Put location of finetuned tokenizer
MeSHGenerating_model = BioGptForCausalLM.from_pretrained('/content/drive/MyDrive/models/BioGPT1000_MeSHTerms_Model') #Put location of finetuned model

for i in range(len(titles)):
    try:
        # GENERATING #
        input_text = "User: " + titles[i] + " Assistant: "
        input_ids = MeSHGenerating_tokenizer.encode(input_text, return_tensors='pt')
        output = MeSHGenerating_model.generate(input_ids, max_length=200)
        generated_text = MeSHGenerating_tokenizer.decode(output[0], skip_special_tokens=True)

        # POST-PROCESSING #
        start_index = generated_text.find("Assistant: ")
        if start_index != -1:
            relevant_text = generated_text[start_index + len("Assistant: "):].strip()
            end_index = relevant_text.find("User:")
            if end_index != -1:
                relevant_text = relevant_text[:end_index].strip()
            model_output_terms = extract_terms(relevant_text)
            model_output_terms = remove_duplicate_terms(model_output_terms)
            for term in model_output_terms:
                mesh_id = fetch_mesh_terms(term, api_key)
                if mesh_id == None:
                  model_output_terms.remove(term)
            model_output_terms_filtered = [term for term in model_output_terms if term]

            # WRITE AND SAVE #
            file_content = f"{model_output_terms_filtered}\n"
            output_filename = f"/content/query_files/{topic_ids[i]}.txt"
            with open(output_filename, "a") as file:
                file.write(file_content)
        else:
            print(f"No Assistant response found for {titles[i]}")

    except Exception as e:
        print(f"Error processing title '{titles[i]}': {e}")

**Generate Keywords**

In [ ]:
api_key = '317014466decf6d06f198a0903107ca95408' #Take this out before you upload
topic_ids, titles = preprocess_titles()

keywords_tokenizer = BioGptTokenizer.from_pretrained('/content/drive/MyDrive/models/BioGPT5000_Keywords') #Put location of finetuned tokenizer
keywords_model = BioGptForCausalLM.from_pretrained('/content/drive/MyDrive/models/BioGPT5000_Keywords') #Put location of finetuned model

for i in range(len(titles)):
    try:
        # GENERATE #
        input_text = "User: " + titles[i] + " Assistant: "
        input_ids = tokenizer.encode(input_text, return_tensors='pt')
        output = model.generate(input_ids, max_length=200)
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

        # POST-PROCESS #
        start_index = generated_text.find("Assistant: ")
        if start_index != -1:
            relevant_text = generated_text[start_index + len("Assistant: "):].strip()
            end_index = relevant_text.find("User:")
            if end_index != -1:
                relevant_text = relevant_text[:end_index].strip()
            model_output_terms = extract_terms(relevant_text)
            model_output_terms = remove_duplicate_terms(model_output_terms)
            model_output_terms_filtered = [term for term in model_output_terms if term]

            # WRITE AND SAVE #
            file_content = f"{model_output_terms_filtered}\n"
            output_filename = f"/content/query_files/{topic_ids[i]}.txt"
            with open(output_filename, "a") as file:
                file.write(file_content)
        else:
            print(f"No Assistant response found for {titles[i]}")

    except Exception as e:
        print(f"Error processing title '{titles[i]}': {e}")

# **Generate Queries**

In [ ]:
def preprocess():
  folder_path = '/content/query_files'
  topic_numbers = []
  titles = []
  MeSHTermsList = []
  keywordsList = []

  try:
      files_in_folder = os.listdir(folder_path)

      for i, filename in enumerate(files_in_folder):
          file_path = os.path.join(folder_path, filename)

          with open(file_path, 'r', encoding='utf-8') as file:
              topic_number = file.readline().strip()
              title = file.readline().strip()
              date = file.readline().strip()
              MeSHTerms = file.readline().strip()
              keywords = file.readline().strip()

              topic_numbers.append(topic_number)
              titles.append(title)
              MeSHTermsList.append(MeSHTerms)
              keywordsList.append(keywords)

  except FileNotFoundError:
      print(f"Folder not found: {folder_path}")
  except Exception as e:
      print(f"An error occurred: {e}")

  return topic_numbers, titles, MeSHTermsList, keywordsList

In [ ]:
tokenizer = BioGptTokenizer.from_pretrained("/content/drive/MyDrive/models/BioGPT_75K_TitlesMeSHKeywords_Generate5/content/output")
model = BioGptForCausalLM.from_pretrained('/content/drive/MyDrive/models/BioGPT_75K_TitlesMeSHKeywords_Generate5/content/output')

topic_ids, titles, MeSHTermsList, keywordsList = preprocess()

for i in range(len(titles)):
    try:
        # GENERATE #
        input_text = "Title: " + titles[i] + " MeSH: " + MeSHTermsList[i] + " Keywords: " + keywordsList[i] + " Query: "
        input_ids = tokenizer.encode(input_text, return_tensors='pt')
        output = model.generate(input_ids, max_length=600)
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

        # POST-PROCESS #
        start_index = generated_text.find("Query: ")
        if start_index != -1:
            relevant_text = generated_text[start_index + len("Query: "):].strip()
            end_index = relevant_text.find("Title:")
            if end_index != -1:
                relevant_text = relevant_text[:end_index].strip()
            relevant_text = relevant_text.replace("Query: ", "")

            # WRITE AND SAVE #
            file_content = f"{relevant_text}\n"
            output_filename = f"/content/query_files/{topic_ids[i]}.txt"
            with open(output_filename, "a") as file:
                file.write(file_content)
        else:
            print(f"No Assistant response found for {titles[i]}")

    except Exception as e:
        print(f"Error processing title '{titles[i]}': {e}")

# **Run Queries and Generate Results File**

**FASS BSLR Tests**

In [ ]:
def ESearch(db, query, date):
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    date = date.replace("-", "/")
    dateRange = "(1900/1/1:" + date + "[pdat])"
    query += " AND " + dateRange

    encodedUrl = urllib.parse.quote(query, safe='')

    params = {
        "term": encodedUrl,
        "usehistory": "y",
        "retmax": "1000",
        "sort": "relevance",
        "db": "pubmed"
    }

    url = base + "esearch.fcgi"

    try:
        return postHTML(url, params)
    except requests.exceptions.RequestException as e:
        print(f"url is: {url}")
        try:
            time.sleep(2)
            print(f"I am sleeping for a second for the following url: {url}")
            return postHTML(url, params)
        except requests.exceptions.RequestException as ex:
            print(f"the following url does not work after sleeping: {url}")
            raise Exception("ERROR after SLEEPING") from ex

def postHTML(urlToRead, params):
    # Initialize variables
    urlParameters = ""
    output = ""

    # Build URL parameters string
    for key, value in params.items():
        if urlParameters == "":
            urlParameters = f"{key}={value}"
        else:
            urlParameters += f"&{key}={value}"

    # Make the POST request
    try:
        response = requests.post(urlToRead, data=urlParameters.encode('utf-8'))

        # Check if request was successful
        response.raise_for_status()

        # Read response content
        output = response.text

    except requests.exceptions.RequestException as e:
        raise e

    return output

def search_to_ids_xml(input_xml):
    output = ""
    test_set = set()

    # Parse the XML
    try:
        tree = ET.parse(StringIO(input_xml))
        root = tree.getroot()

        # Find all "Id" elements
        for id_elem in root.findall('.//Id'):
            id_str = id_elem.text
            if id_str and id_str.strip() and id_str not in test_set:
                test_set.add(id_str)
                output += id_str + ","

    except ET.ParseError as ex:
        print(f"Parse error: {ex}")
    except Exception as ex:
        print(f"An error occurred: {ex}")

    return output

def get_output(x, slr_id, method):
    tokens = x.split(',')
    queries = []
    unique_tokens = set()
    output = ""
    count = 0
    query = ""

    for token in tokens:
        if token and token not in unique_tokens:
            unique_tokens.add(token)
            if count == 199:
                queries.append(query)
                query = ""
                count = 0
            count += 1
            query += token + ","

    if query:
        queries.append(query)

    for q in queries:
        try:
            ids = pmc_to_doi(q)
            ranking = 1
            for id_ in ids:
                score = 1.0 / ranking
                output += f"{slr_id} 0 {id_} {ranking} {score:.6f} {method}\n"
                ranking += 1

        except Exception as ex:
            print(f"An error occurred: {ex}")

    return output

def find_all_dois(xml, pmc):
    results = []
    pmid_doi = {}
    ids = pmc.split(",")

    root = ET.fromstring(xml)
    for record in root.findall(".//record"):
        pmid = record.get("pmid")
        doi = record.get("doi")
        error = record.get("error")
        if doi:
            pmid_doi[pmid] = doi

    for id in ids:
        if id in pmid_doi:
            results.append(pmid_doi[id])

    return results

def pmc_to_doi(pmc):
    base_url = "https://www.ncbi.nlm.nih.gov/pmc/utils/idconv/v1.0/"
    full_url = f"{base_url}?ids={pmc}"
    response = get_html(full_url)
    return find_all_dois(response, pmc)

def doi_error(pmc):
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=pubmed&id="
    full_url = f"{base_url}{pmc}&rettype=xml"
    response = requests.get(full_url)
    response.raise_for_status()

    # Parse the XML data
    root = ET.fromstring(response.content)

    # Find the DOI
    doi = None
    for eloc in root.findall('.//ELocationID'):
        if eloc.attrib.get('EIdType') == 'doi' and eloc.attrib.get('ValidYN') == 'Y':
            doi = eloc.text
            break

    if doi:
      return (doi)
    else:
      return None

def get_html(url_to_read, params=None):
    result = ""
    if params:
        response = requests.get(url_to_read, params=params)
    else:
        response = requests.get(url_to_read)
    response.raise_for_status()
    result = response.text
    return result

def FASS_process_files(folder_path, results_file_path):
    with open(results_file_path, 'w') as result_file:
        for filename in os.listdir(folder_path):
            if filename.endswith(".txt"):
                file_path = os.path.join(folder_path, filename)
                with open(file_path, 'r') as file:
                    lines = file.readlines()
                    if len(lines) >= 3:
                        query_line = lines[5].strip()  # The 4th line in the file should be the query, change this if not
                        query = query_line.replace("Based on the following SLR title, please provide 5 complex pubmed Entrez formatted queries without descriptions, in plain text, not exceeding 15 query terms, such that they may be used directly on Pubmeds website. ", "")
                        query = query.split('2. ')[0].strip()
                        query = query.replace("1. ", "")
                        #query_line = query_line.replace("1. ", "")
                        id = lines[0].strip()
                        title = lines[1].strip()
                        dateline = lines[2].strip()
                        date_only = dateline.split()[0]
                        query = query.replace("\"", "") # Remove quotes from the generated queries

                        try:
                          results = search_to_ids_xml(ESearch("pubmed", query, date_only))
                          output = get_output(results, id, "BioGPT")
                        except:
                          output = ""

                        time.sleep(1)

                        result_file.write(output)

results_file_path = '/content/cgtrun5_quotes.txt' # Update name with results file name
FASS_process_files('/content/query_files', results_file_path) # Update path with where query files are stored

**CLEF TAR Tests**

In [ ]:
def fetch_pmids(query, end_date, id, max_results=1000):
    Entrez.email = "leandra.budau@torontomu.ca"
    Entrez.api_key = "0c185c170813f85399fe02d033394e8de208"
    date_range = f"(1900/1/1:{end_date}[pdat])"
    formatted_query = f'{query} AND {date_range}'

    handle = Entrez.esearch(db="pubmed", term=formatted_query, retmax=1000, sort="Relevance")
    record = Entrez.read(handle)
    handle.close()

    if len(record["IdList"])==0:
      print(id)
      print(query)

    return record["IdList"]

def CLEF_process_files(folder_path, results_file_path):
    with open(results_file_path, 'w') as result_file:
        for filename in os.listdir(folder_path):
            if filename.endswith(".txt"):
                file_path = os.path.join(folder_path, filename)
                with open(file_path, 'r') as file:
                    lines = file.readlines()
                    if len(lines) >= 5:
                        query_line = lines[4].strip()  # The query should be on the 3rd line of the file. If not, change this.
                        query_line = query_line.split('2. ')[0].strip()
                        query_line = query_line.replace("1. ", "")
                        id = lines[0].strip()
                        query_line = query_line.replace("\"", "")
                        print(query_line)

                        if (filename.replace(".txt", "") in Testing_Set):
                          end_date = "2017/03/8"
                        elif (filename.replace(".txt", "") in Training_Set):
                          end_date = "2018/04/27"
                        try:
                          query = query_line.replace("Based on the following SLR title, please provide 5 complex pubmed Entrez formatted queries without descriptions, in plain text, such that they may be used directly on Pubmed s website. ", "")
                          pmids = fetch_pmids(query, end_date, id)
                        except:
                          time.sleep(3)
                        time.sleep(3)

                        topic_id = filename.replace('.txt', '')
                        rank = 1
                        for pmid in pmids:
                            score = 1 / rank
                            result_file.write(f"{topic_id} 0 {pmid} {rank} {score:.8f} TitlesOnlyChatGPT\n")
                            rank += 1

def filter_results(qrel_path, results_path, output_path):
    qrel_data = {}
    with open(qrel_path, 'r') as file:
        for line in file:
            parts = line.split()
            task_number = parts[0]
            pmid = parts[2]
            if task_number not in qrel_data:
                qrel_data[task_number] = set()
            qrel_data[task_number].add(pmid)

    with open(results_path, 'r') as results_file, open(output_path, 'w') as output_file:
        for line in results_file:
            parts = line.split()
            if len(parts)>0:
              task_number = parts[0]
              pmid = parts[2]
              if (task_number in qrel_data and pmid in qrel_data[task_number]) or pmid=='X':
                  output_file.write(line)


results_file_path = '/content/ChatGPT-PE_CLEFTAR.txt' # Update name with results file name
CLEF_process_files('/content/query_files', results_file_path) # Update path with where query files are stored
filter_results('/content/total_72doc_qrel.txt', '/content/ChatGPT-PE_CLEFTAR.txt', '/content/ChatGPT-PE_CLEFTAR_revised.txt') #Update names of comparative qrel and results file